In [1]:
import pandas as pd
from collections import Counter
from rapidfuzz import fuzz
import numpy as np
from tqdm import tqdm
import re
import ast
from utils.utils_oab import score_fuzz

In [2]:
mte_clt = pd.read_csv('csvs/mte/mte_formacao_questions_unified_no_answer.csv')
# Options as lists: the match compares the options too
mte_clt['options'] = mte_clt['options'].apply(ast.literal_eval)

In [3]:
mte_all = pd.read_csv('csvs/mte/mte_formacao_questions_all.csv')

In [4]:
mte_all = mte_all.drop_duplicates()
mte_all = mte_all.reset_index(drop=True)
mte_all['options'] = mte_all['options'].apply(ast.literal_eval)

In [5]:
match_dict = {}
dict_count = []
for i,r in mte_clt.iterrows():
    # Same statement and same options: some statements repeat across exams with other options
    mask = mte_all[(mte_all['question'] == r['question']) & (mte_all['options'].apply(lambda o: o == r['options']))]
    if len(mask) == 0:
        match_dict[r['question_number']] = None
        dict_count.append(0)
    if len(mask) == 1:
        match_dict[r['question_number']] = mask.iloc[0]
        dict_count.append(1)
    if len(mask) > 1:
        match_dict[r['question_number']] = 'Dupla ocorrência'
        dict_count.append(2)

In [6]:
for i, r in tqdm(mte_clt.iterrows()):
    if match_dict[i+1] is None:
        score = score_fuzz(r['question'], mte_all, options=r['options'])

        if len(score) == 1:
            match_dict[i+1] = mte_all.iloc[score[0]]
            dict_count[i] = 1

        elif len(score) > 1:
            match_dict[i+1] = 'Dupla ocorrência'
            dict_count[i] = 2

        else:
            match_dict[i+1] = None

78it [00:00, 10278.86it/s]


In [7]:
dict_count = []
for i in match_dict.values():
    if i is None:
        dict_count.append(0)
    elif type(i) == pd.Series:
        dict_count.append(1)
    else:
        dict_count.append(2)

In [8]:
match_dict

{1: question_number                                                   44
 question           A respeito das prerrogativas e dos deveres dos...
 options            [As normas coletivas do trabalho têm prazo ind...
 exam               Curso de Formação - Auditor Fiscal do Trabalho...
 answer                                                             D
 Name: 103, dtype: object,
 2: question_number                                                   43
 question           A respeito da evolução histórica do direito co...
 options            [A CLT foi o primeiro documento político e jur...
 exam               Curso de Formação - Auditor Fiscal do Trabalho...
 answer                                                             B
 Name: 102, dtype: object,
 3: question_number                                                   42
 question           Assinale a opção correspondente a um exemplo d...
 options            [vale transporte, adicional de insalubridade, ...
 exam               Curso d

In [9]:
Counter(dict_count)

Counter({1: 78})

In [10]:
for i in match_dict.keys():
    if match_dict[i] is None:
        print(f"Question {i} not found")

In [11]:
mte_with_answer = []
for i in match_dict.keys():
    if type(match_dict[i]) == pd.Series:
        mte_with_answer.append({
            'question_number': int(match_dict[i]['question_number']),
            'question': match_dict[i]['question'],
            'options': match_dict[i]['options'],
            'answer': match_dict[i]['answer'],
            'exam': match_dict[i]['exam']
        })

In [ ]:
df_mte_with_answer = pd.DataFrame(mte_with_answer)
# Annulled questions (without answer in the answer key) are not saved
annulled = df_mte_with_answer[df_mte_with_answer['answer'].isna()]
df_mte_with_answer = df_mte_with_answer[df_mte_with_answer['answer'].notna()].reset_index(drop=True)
print(f'Annulled questions removed: {len(annulled)}')
for exam, numbers in annulled.groupby('exam')['question_number']:
    print(f'  {exam}: {numbers.tolist()}')

In [13]:
df_mte_with_answer.to_csv('csvs/mte/mte_formacao_questions_with_answer.csv', index=False)

In [ ]:
# All MTE questions: ESAF exams + Curso de Formação
mte_esaf = pd.read_csv('csvs/mte/mte_esaf_questions_all.csv')
columns = ['question_number', 'question', 'options', 'exam', 'answer']
mte_questions_all = pd.concat([mte_esaf[columns], df_mte_with_answer[columns]], ignore_index=True)
mte_questions_all.to_csv('csvs/mte/mte_questions_all.csv', index=False)
mte_questions_all['exam'].value_counts()